In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report, f1_score, accuracy_score, recall_score, precision_score
import seaborn as sns
from pathlib import Path
import os

In [12]:
sds_dir = Path('/Users/lukashat/sds_mount/sds/sd22c003/phenotyping_benchmark/visual_qc/corrected_annotations/')
dataset = 'cHL_2_MIBI'

In [13]:
dfs = []
for csv in os.listdir(os.path.join(sds_dir, dataset)):
    if csv.endswith('.csv'):
        df = pd.read_csv(os.path.join(sds_dir, dataset, csv))
        dfs.append(df)
full_df = pd.concat(dfs, ignore_index=True)

In [16]:
full_df

,CD45,CD20,dsDNA,pSLP-76,SLP-76,anti-H2AX (pS139),CD163,Histone H3,CD45RO,CD28,...,y,sample_id,level_1_cell_type,level_2_cell_type,cell_type,probability,NewAnnotation_Lukas,Reannotated,_is_dummy,cell_id
0,0.000000,0.005460,0.000247,0.0,0.0,0.000000,0.000000,0.222539,0.000332,0.000000,...,1261.0,1.csv,Immune,Lymphoid_immune,B_cell,1.0,NaN,False,False,23586
1,0.000000,0.001577,0.000000,0.0,0.0,0.000000,0.000000,0.208911,0.000051,0.000000,...,1261.0,1.csv,Immune,Lymphoid_immune,B_cell,1.0,NaN,False,False,23616
2,0.000000,0.000276,0.000000,0.0,0.0,0.000000,0.000000,0.147267,0.000000,0.000000,...,1262.0,1.csv,Immune,Lymphoid_immune,NK_cell,1.0,NaN,False,False,23639
3,0.000000,0.007751,0.000000,0.0,0.0,0.000000,0.000000,0.154699,0.000124,0.000000,...,1263.0,1.csv,Immune,Lymphoid_immune,B_cell,1.0,NaN,False,False,23670
4,0.000000,0.014449,0.000000,0.0,0.0,0.000000,0.000000,0.212461,0.000221,0.000000,...,1266.0,1.csv,Immune,Lymphoid_immune,B_cell,1.0,NaN,False,False,23681
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
897,0.000981,0.000000,0.000000,0.0,0.0,0.000469,0.144613,0.100192,0.001510,0.000000,...,1984.0,3.csv,Immune,Myeloid_immune,M2_Macrophage,1.0,NaN,False,False,17825
898,0.001059,0.000000,0.000000,0.0,0.0,0.000000,0.017144,0.151911,0.000000,0.000000,...,1986.0,3.csv,Immune,Myeloid_immune,M2_Macrophage,1.0,NaN,False,False,17834
899,0.001191,0.000273,0.002276,0.0,0.0,0.000187,0.001297,0.113112,0.000064,0.000006,...,1988.0,3.csv,Immune,Lymphoid_immune,CD4+_T_cell,1.0,NaN,False,False,17877
900,0.000000,0.000000,0.000000,0.0,0.0,0.000000,0.004354,0.164960,0.000000,0.000000,...,1990.0,3.csv,unedfined,undefined,undefined,1.0,NaN,False,False,17878


In [17]:
# Expert review accuracy
# All cells were reviewed. Reannotated==True means expert changed the label.
# NewAnnotation_Lukas holds the correction; NaN means expert agreed with original.

orig_col = 'cell_type'
new_col  = 'NewAnnotation_Lukas'
flag_col = 'Reannotated'

df = full_df.copy()
df['expert_label'] = df.apply(
    lambda r: r[new_col] if r[flag_col] else r[orig_col], axis=1
)

overall_acc = (~df[flag_col]).mean()
print(f"Overall agreement (original GT vs expert): {overall_acc:.1%}  "
      f"({(~df[flag_col]).sum()} / {len(df)} cells)\n")

# Per-original-cell-type: how many were changed
per_type = (
    df.groupby(orig_col)[flag_col]
    .agg(n_cells='count', n_changed='sum')
    .assign(pct_changed=lambda d: d.n_changed / d.n_cells * 100)
    .sort_values('pct_changed', ascending=False)
)
print("Changes per original cell type:")
print(per_type.to_string())

# Confusion: original → expert correction (only changed cells)
changed = df[df[flag_col]]
if not changed.empty:
    print(f"\nOriginal → Expert correction (n={len(changed)}):")
    print(changed.groupby([orig_col, new_col]).size().rename('count').reset_index().to_string(index=False))


Overall agreement (original GT vs expert): 91.4%  (824 / 902 cells)

Changes per original cell type:
                n_cells  n_changed  pct_changed
cell_type                                      
Cancer               48         14    29.166667
Dendritic_cell       89         16    17.977528
CD4+_T_cell          95         15    15.789474
NK_cell              15          2    13.333333
M1_Macrophage        12          1     8.333333
Treg                150         12     8.000000
undefined            58          4     6.896552
M2_Macrophage        88          5     5.681818
CD8+_T_cell          87          3     3.448276
B_cell              157          4     2.547771
Neutrophil           89          2     2.247191
Endothelial          14          0     0.000000

Original → Expert correction (n=78):
     cell_type NewAnnotation_Lukas  count
        B_cell         CD4+_T_cell      1
        B_cell      Dendritic_cell      2
        B_cell           undefined      1
   CD4+_T_cell       